In [2]:
import torch
from transformers import AutoModel
from plm_models import ESM2
esm2_8m_checkpoint = "/data/luyq/esm2_models/esm2_t6_8M_UR50D"
# model = ESM2(esm2_checkpoint=esm2_8m_checkpoint, head_type='3MLP', fine_tune=False)
esm8m =  AutoModel.from_pretrained(esm2_8m_checkpoint)
for name, param in esm8m.named_parameters():
    print(name, ":", param.size())

data = torch.randint(0, 20, (2, 30))    # batch_size=2, seq_len=30
output = esm8m(data)
print(output.last_hidden_state.size())  # (2, 30, 320)

Some weights of EsmModel were not initialized from the model checkpoint at /data/luyq/esm2_models/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


embeddings.word_embeddings.weight torch.Size([33, 320])
embeddings.position_embeddings.weight torch.Size([1026, 320])
encoder.layer.0.attention.self.query.weight torch.Size([320, 320])
encoder.layer.0.attention.self.query.bias torch.Size([320])
encoder.layer.0.attention.self.key.weight torch.Size([320, 320])
encoder.layer.0.attention.self.key.bias torch.Size([320])
encoder.layer.0.attention.self.value.weight torch.Size([320, 320])
encoder.layer.0.attention.self.value.bias torch.Size([320])
encoder.layer.0.attention.output.dense.weight torch.Size([320, 320])
encoder.layer.0.attention.output.dense.bias torch.Size([320])
encoder.layer.0.attention.LayerNorm.weight torch.Size([320])
encoder.layer.0.attention.LayerNorm.bias torch.Size([320])
encoder.layer.0.intermediate.dense.weight torch.Size([1280, 320])
encoder.layer.0.intermediate.dense.bias torch.Size([1280])
encoder.layer.0.output.dense.weight torch.Size([320, 1280])
encoder.layer.0.output.dense.bias torch.Size([320])
encoder.layer.0.L

In [3]:
print(type(output))
print(output.keys())

<class 'transformers.modeling_outputs.BaseModelOutputWithPoolingAndCrossAttentions'>
odict_keys(['last_hidden_state', 'pooler_output'])


In [5]:
mean_output = output[0].mean(dim=1)  # [batch_size, hidden_size]
# output[0].size()  # (2, 30, 320)
mean_output == output[1]
# output[1].size()  # (2, 320)

tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, F

In [15]:
import pandas as pd
mic_data = "/home/luyq/PLM_AMP_Regression/data/mic_data_b.csv"
mic = pd.read_csv(mic_data)
mic.isna().sum()

sequence       0
value          0
labels      4979
dtype: int64

In [16]:
mic["sequence"].describe()

count                                  6083
unique                                 4546
top       LDDLPGALSELSDLHAHKLRVDPVNFKLLSHSL
freq                                      2
Name: sequence, dtype: object

In [17]:
mic["seq_len"] = mic["sequence"].apply(lambda x: len(x))
mic["seq_len"].describe()

count    6083.000000
mean       20.665626
std        12.775899
min         2.000000
25%        13.000000
50%        18.000000
75%        25.000000
max       190.000000
Name: seq_len, dtype: float64

In [25]:
# 针对 sequence 和 value 两列去重，并找到最长的 sequence
mic_unique = mic.drop_duplicates(subset=["sequence", "value"])
longest_seq = mic_unique.loc[mic_unique['sequence'].str.len().idxmax(), 'sequence']
print(f"唯一样本数: {len(mic_unique)}")
print(f"最长的序列长度: {len(longest_seq)}")
print(f"最长的序列: {longest_seq}")

唯一样本数: 4546
最长的序列长度: 190
最长的序列: MTESLVLSPAPAKPKRVKASRRSASHPTYSEMIAAAIRAEKSRGGSSRQSIQKYIKSHYKVGHNADLQIKLSIRRLLAAGVLKQTKGVGASGSFRLAKSDKAKRSPGKKKKAVRRSTSPKKAARPRKARSPAKKPKATARKARKKSRASPKKAKKPKTVKAKSRKASKAKKVKRSKPRAKSGARKSPKKK


In [26]:

mic_unique = mic_unique[mic_unique["seq_len"] <= 60]
mic_unique = mic_unique[["sequence","value"]] 
mic_unique

,sequence,value
0,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2.099950
1,AAAKAALNAVLVGANA,1.903090
2,AAGRYQLLSRYWDAYR,2.176091
3,AAKHAAHRA,2.729535
4,AAKIILNPKFRCFAAFC,2.204120
...,...,...
6074,YSWPRMPRIPRLPRYPRYPRYPRWPRWPRQPTIYA,0.301030
6079,YVPLPNVPQPGRRPFPTFPGQGPFNPKIKWPQGY,0.548456
6080,YVPPVQKPHPNGPKFPTFP,1.364542
6081,YWKKWKKLRRIFMLV,-0.020287


In [27]:
mic_unique.to_csv("/home/luyq/PLM_AMP_Regression/data/mic_data_unique.csv", index=False)

In [28]:
# 按6:2:2划分mic_unique为train/valid/test，并保存为csv
from sklearn.model_selection import train_test_split
import os
output_dir = "/home/luyq/PLM_AMP_Regression/data/mic_data_unique"
os.makedirs(output_dir, exist_ok=True)
train, temp = train_test_split(mic_unique, test_size=0.4, random_state=42)
valid, test = train_test_split(temp, test_size=0.5, random_state=42)
train.to_csv(os.path.join(output_dir, "train.csv"), index=False)
valid.to_csv(os.path.join(output_dir, "valid.csv"), index=False)
test.to_csv(os.path.join(output_dir, "test.csv"), index=False)
print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 2674, valid: 891, test: 892
